In [ ]:
import torch 
import math


class TokenEmbeddings:
    def __init__(self,vocab_size,num_dims):
        self.weights = torch.randn(vocab_size,num_dims)
        
    def forward(self,indices):
        self.indices = indices
        return self.weights[indices]

    def backward(self,grad_out):
        for i,row_idx in enumerate(self.indices):
            self.weights.grad[row_idx]+=grad_out[i]

    def parameters(self):
        return [self.weights]


class PositionalEmbeddings:
    def __init__(self,seq_len,num_dims):
        self.weights = torch.randn(seq_len,num_dims)

    def forward(self,positions):
        self.positions = positions
        return self.weights[positions]

    def backward(self,grad_out):
        for i,pos in enumerate(self.positions):
            self.weights.grad[pos]+=grad_out[i]

    def parameters(self):
        return [self.weights]

    
class LinearLayer:
    
    def __init__(self, in_features, out_features, bias=True):
        self.weights = torch.randn(out_features,in_features)
        self.has_bias = bias
        if bias:
            self.bias = torch.zeros(out_features)
        else:
            self.bias = None

    def forward(self,x):
        self.x = x
        out = x @ self.weights.T
        if self.has_bias and self.bias is not None:
            out = out + self.bias
        return out

    def backward(self,grad_out):
        grad_inputs = grad_out @ self.weights
        self.weights.grad = grad_out.T @ self.x
        self.bias.grad = grad_out.sum(dim=0)
        return grad_inputs


    
class CausalSelfAttention:
    def __init__(self,num_dims,num_heads):
        self.num_dims = num_dims
        self.num_heads = num_heads
        self.head_dims = num_dims // num_heads

        self.w_q = LinearLayer(num_dims,num_dims,bias=False)
        self.w_k = LinearLayer(num_dims,num_dims,bias=False)
        self.w_v = LinearLayer(num_dims,num_dims,bias=False)

        self.proj_out = LinearLayer(num_dims,num_dims)

    def softmax(self,scores):
        exp_scores = torch.exp(scores)
        self.sum_exp = exp_scores.sum(dim=-1,keepdim=True)
        out = exp_scores / self.sum_exp
        return out
        
    def forward(self,x):
        B,T,D = x.shape

        Q = self.w_q.forward(x)
        K = self.w_k.forward(x)
        V = self.w_v.forward(x)

        Q  = Q.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        K  = K.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        V =  V.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.head_dims)
        masks = torch.triu(torch.ones(T,T,dtype=torch.bool),diagonal=1)
        scores = scores.masked_fill(masks, float("-inf"))

        attn_scores = self.softmax(scores)

        out = attn_scores @ V
        out = out.transpose(1,2).contiguous().view(B,T,D)

        out = self.proj_out(out)
        return out


    def backward(self):
        pass

        
    def parameters(self):
        return [self.w_q.weights, self.w_k.weights, self.w_v.weights, self.proj_out.weights, self.proj_out.bias]
        
class Relu:
    def forward(self,x):
        self.x = x
        return torch.clamp(x,min=0)

    def backward(self,grad_out):
        return grad_out * (self.x >0)


class MyMlp:
    def __init__(self,in_features,hidden_features,out_features):
        self.linear1 = LinearLayer(in_features,hidden_features)
        self.relu = Relu()
        self.linear2 = LinearLayer(hidden_features,out_features)

    def forward(self,x):
        x = self.linear1.forward(x)
        x = self.relu.forward(x)
        x = self.linear2.forward(x)
        return x

    def backward(self,grad_out):
        grad_linear2 = self.linear2.backward(grad_out)
        grad_relu = self.relu.backward(grad_linear2)
        grad_linear1 = self.linear1.backward(grad_relu)
        return grad_linear1

    def parameters(self):
        return [ self.linear1.weights,self.linear1.bias,self.linear2.weights,self.linear2.bias]


class MyMseLoss:
    def forward(self,y_pred,y_true):
        self.y_pred = y_pred
        self.y_true = y_true
        return torch.mean((self.y_pred-self.y_true)**2)

    def backward(self):
        N = self.y_pred.numel()
        grad_out = (2/N)*(self.y_pred-self.y_true)
        return grad_out


class Optim:
    def __init__(self,params,lr=0.01):
        self.params = params
        self.lr = lr

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data-=p.grad*self.lr



      
